# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, subprocess

if not os.path.isdir("Flyranks-internship-assignmnet-1"):
    subprocess.run(["git", "clone", "--depth", "1",
        "https://github.com/MaryamNaveed-bioinfo/Flyranks-internship-assignmnet-1"], check=True)
os.chdir("Flyranks-internship-assignmnet-1")
print("Working dir:", os.getcwd())

subprocess.run(["pip", "install", "-q", "duckdb"], check=True)

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Ready to query:", REL)

Working dir: /content/Flyranks-internship-assignmnet-1
Connected. Ready to query: hf://datasets/FlyRank/internship-warehouse


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
top_level = sorted(set(f.split("/")[0] for f in files))
print(top_level)

['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance', 'fact_content_daily_performance_sample.parquet', 'fact_content_query_90d.parquet']


**Signal check 1 : Staleness (linked to FlyRank's "refresh" flag logic)**

Hypothesis: pages that haven't been updated in a long time are more likely to be declining.

In [3]:
q_stale = f"""
WITH content_age AS (
    SELECT
        content_hash_id,
        content_updated_date,
        DATE_DIFF('day', content_updated_date, CURRENT_DATE) AS days_since_update
    FROM read_parquet('{REL}/dim_content.parquet')
),
joined AS (
    SELECT
        f.content_hash_id,
        f.gsc_impressions,
        f.gsc_clicks,
        c.days_since_update,
        CASE
            WHEN c.days_since_update < 90 THEN '0-89 days'
            WHEN c.days_since_update < 180 THEN '90-179 days'
            WHEN c.days_since_update < 365 THEN '180-364 days'
            ELSE '365+ days'
        END AS staleness_bucket
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN content_age c ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_data_available IS TRUE
)
SELECT
    staleness_bucket,
    COUNT(*) AS n,
    AVG(CASE WHEN gsc_impressions > 0 AND gsc_clicks IS NOT NULL
             THEN gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) END) AS avg_ctr
FROM joined
GROUP BY staleness_bucket
ORDER BY staleness_bucket
"""
con.sql(q_stale).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_ctr
0,0-89 days,2825203,0.003269
1,180-364 days,13864,0.009181
2,365+ days,769,0.001390
3,90-179 days,771225,0.002285


**Verdict: MIXED** : CTR does not decline consistently with staleness. The 0-89 and
90-179 day buckets show a mild expected decline (0.33% -> 0.23%), but the 180-364 day
bucket spikes to 0.92% CTR (nearly 3x higher), before dropping again at 365+ days
(0.14%). This non-monotonic pattern, combined with very small and uneven sample sizes
in the older buckets (13,864 and 769 rows vs. 2.8M and 771K in the fresher buckets),
means staleness alone is not a reliable signal here.

**Signal check 2 — Position vs CTR (linked to FlyRank's "CTR-fix" flag logic)**

Hypothesis: pages ranking in worse positions get lower CTR if true, this supports
flagging low-CTR-for-position pages as review candidates.

In [4]:
q_position = f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN '1-3 (top)'
        WHEN gsc_avg_position <= 10 THEN '4-10'
        WHEN gsc_avg_position <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,
    COUNT(*) AS n,
    AVG(CASE WHEN gsc_impressions > 0 AND gsc_clicks IS NOT NULL
             THEN gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) END) AS avg_ctr
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
GROUP BY position_bucket
ORDER BY position_bucket
"""
con.sql(q_position).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_ctr
0,1-3 (top),727362,0.004756
1,11-20,519223,0.002770
2,21+,908354,0.001289
3,4-10,1456122,0.003473


**Verdict: CONFIRMED** : Average CTR declines cleanly and consistently as position gets
worse: from 0.48% in the top 3 positions down to 0.13% at position 21+. Backed by large,
stable sample sizes in every bucket (519K–1.46M rows).

**My rule:** Flag a page as needing review if it ranks in a strong position (top 20) but
still shows meaningfully low CTR relative to its position bucket this combines a
CONFIRMED signal (position) with real visibility (impressions), rather than leaning on
the MIXED staleness signal. Staleness is a secondary, manual-check factor only.

**Reason codes:**
- `ctr_review_candidate`  position ≤ 20 AND CTR below the bucket's expected average
  AND impressions ≥ 100
- `no_flag` does not meet the above criteria

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Approach:** For each page, compare its actual CTR to the expected CTR for other pages
in the same position bucket. Pages that under-perform their bucket's expected CTR and
have enough traffic for that gap to matter get flagged and scored.

In [5]:
q_queue = f"""
WITH page_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
bucketed AS (
    SELECT
        *,
        gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr,
        CASE
            WHEN gsc_avg_position <= 3 THEN '1-3 (top)'
            WHEN gsc_avg_position <= 10 THEN '4-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            ELSE '21+'
        END AS position_bucket
    FROM page_agg
    WHERE gsc_impressions >= 100
),
bucket_avg AS (
    SELECT position_bucket, AVG(ctr) AS expected_ctr
    FROM bucketed
    GROUP BY position_bucket
)
SELECT
    b.client_hash_id,
    b.content_hash_id,
    b.gsc_impressions,
    b.gsc_clicks,
    b.gsc_avg_position,
    b.position_bucket,
    b.ctr,
    a.expected_ctr,
    (a.expected_ctr - b.ctr) AS ctr_gap,
    b.gsc_impressions * (a.expected_ctr - b.ctr) AS score,
    CASE
        WHEN b.gsc_avg_position <= 20 AND b.ctr < a.expected_ctr AND b.gsc_impressions >= 100
        THEN 'ctr_review_candidate'
        ELSE 'no_flag'
    END AS reason_code,
    CASE
        WHEN b.gsc_avg_position <= 20 AND b.ctr < a.expected_ctr AND b.gsc_impressions >= 100
        THEN 'review_title_and_meta'
        ELSE 'monitor'
    END AS action_label
FROM bucketed b
JOIN bucket_avg a ON b.position_bucket = a.position_bucket
WHERE b.gsc_avg_position <= 20 AND b.ctr < a.expected_ctr AND b.gsc_impressions >= 100
ORDER BY score DESC
"""
queue = con.sql(q_queue).df()
print(f"Queue size: {len(queue)} pages flagged")
queue.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue size: 50570 pages flagged


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,position_bucket,ctr,expected_ctr,ctr_gap,score,reason_code,action_label
0,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,4-10,0.000113,0.003228,0.003115,661.720669,ctr_review_candidate,review_title_and_meta
1,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,1-3 (top),0.001420,0.003633,0.002213,450.253719,ctr_review_candidate,review_title_and_meta
2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,1.0,4.545582,4-10,0.000007,0.003228,0.003221,434.779547,ctr_review_candidate,review_title_and_meta
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,3.219473,4-10,0.000301,0.003228,0.002928,418.719574,ctr_review_candidate,review_title_and_meta
4,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,1.0,9.385150,4-10,0.000008,0.003228,0.003220,399.561157,ctr_review_candidate,review_title_and_meta
5,client_62f4a7e64f5e0096,content_7c6373141eae744a,132593.0,83.0,5.789019,4-10,0.000626,0.003228,0.002602,345.060492,ctr_review_candidate,review_title_and_meta
6,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,15.0,9.536301,4-10,0.000139,0.003228,0.003089,332.321955,ctr_review_candidate,review_title_and_meta
7,client_62f4a7e64f5e0096,content_acbcc847f8996314,170808.0,262.0,3.361195,4-10,0.001534,0.003228,0.001694,289.433005,ctr_review_candidate,review_title_and_meta
8,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,89332.0,4.0,7.786219,4-10,0.000045,0.003228,0.003184,284.397576,ctr_review_candidate,review_title_and_meta
9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,194337.0,361.0,4.450106,4-10,0.001858,0.003228,0.001371,266.393541,ctr_review_candidate,review_title_and_meta


In [6]:
import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Saved:", "work/outputs/baseline_action_score.csv")
print("Rows written:", len(queue))

Saved: work/outputs/baseline_action_score.csv
Rows written: 50570


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top-20 review : action, reason code, confidence, what would prove it wrong**

1. `content_44f34c0a90047651` (client_23a62021009f63c4) 212K impr, 24 clicks, pos 7.3, CTR 0.011% vs 0.32%.
   **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: HIGH** (large impressions, real click volume, clean gap)
   **Would be wrong if:** low CTR reflects a misleading title that rightly gets skipped fix would be relevance, not wording.

2. `content_8d7d99f109e19aa2` (client_e547b89c05043229) 203K impr, 289 clicks, pos 2.56 (top 3), CTR 0.14% vs 0.36%.
   **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: HIGH** (top-3 rank, high click count makes this stable)
   **Would be wrong if:** this is a branded/navigational query where users already know the destination.

3. `content_8e1334d6356668e3` (client_73cda7b4e4f265ea) 135K impr, **1 click**, pos 4.5, CTR 0.0007% vs 0.32%.
   **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: LOW** (1 click one accidental click either way would swing CTR completely)
   **Would be wrong if:** it's a technical/indexing issue, not a title/meta problem needs audit before content fix.

4. `content_34a70fea29d15f24` (client_62f4a7e64f5e0096) 143K impr, 43 clicks, pos 3.2, CTR 0.03% vs 0.32%.
   **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: HIGH** (good position, real click count, large gap)
   **Would be wrong if:** a featured snippet is suppressing organic clicks regardless of title.

5. `content_fec55986a1868d62` (client_73cda7b4e4f265ea) 124K impr, **1 click**, pos 9.4, CTR 0.0008% vs 0.32%.
   **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: LOW** (1 click, same fragility as #3, same client as #3)
   **Would be wrong if:** same root technical cause as #3 rather than 2 separate problems.

6. `content_7c6373141eae744a` (client_62f4a7e64f5e0096) 133K impr, 83 clicks, pos 5.8, CTR 0.06% vs 0.32%.
   **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: HIGH** (solid click count, consistent underperformance)
   **Would be wrong if:** heavy paid-ad or snippet competition on this keyword, not a title problem.

7. `content_f6116743b00afc2d` (client_62f4a7e64f5e0096) 108K impr, 15 clicks, pos 9.5, CTR 0.01% vs 0.32%.
   **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: MEDIUM** (15 clicks is thin, though impressions are large)
   **Would be wrong if:** this client's whole page template is the real cause, not this one page.

8. `content_acbcc847f8996314` (client_62f4a7e64f5e0096) 171K impr, 262 clicks, pos 3.36, CTR 0.15% vs 0.32%.
   **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: MEDIUM** (smallest gap in top 10  flagged mainly by volume, not severity)
   **Would be wrong if:** 0.15% is actually normal for this query type, making the bucket average too broad a baseline.

9. `content_cd3d932d4e1c8db0` (client_9958f0a7ae1df715)  89K impr, 4 clicks, pos 7.8, CTR 0.004% vs 0.32%.
   **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: LOW** (4 clicks thin, but different client confirms this isn't purely a client_62f4... artifact)
   **Would be wrong if:** page is new and CTR hasn't stabilized yet.

10. `content_b99ea6861864dea5` (client_62f4a7e64f5e0096) 194K impr, 361 clicks, pos 4.45, CTR 0.19% vs 0.32%.
    **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: HIGH** (largest click count in the list very stable estimate)
    **Would be wrong if:** this client's overall volume is just larger, so they show up more often regardless of real underperformance.

11. `content_046fc480045b88f5` (client_a80fca3f171ed1de) 84K impr, 6 clicks, pos 7.29, CTR 0.007% vs 0.32%.
    **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: LOW** (6 clicks, fragile)
    **Would be wrong if:** small-sample noise rather than a real content problem.

12. `content_f43118e089ecc69a` (client_73cda7b4e4f265ea) 139K impr, 191 clicks, pos 5.04, CTR 0.14% vs 0.32%.
    **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: HIGH** (good click volume, third page from this client reinforces a client pattern)
    **Would be wrong if:** this client's traffic volume alone explains their frequent appearance.

13. `content_306bc78dff1eb683` (client_e547b89c05043229) 81K impr, 35 clicks, pos 1.49 (top 3), CTR 0.04% vs 0.36%.
    **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: HIGH** (top-3 position with very low CTR is a strong, clear signal)
    **Would be wrong if:** it's a branded query, same caveat as #2 from the same client.

14. `content_9540d884af3e41fd` (client_a80fca3f171ed1de) 82K impr, 11 clicks, pos 7.79, CTR 0.013% vs 0.32%.
    **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: MEDIUM** (11 clicks, thin but not negligible)
    **Would be wrong if:** small-sample noise, same as #11 from the same client may be a client-level pattern.

15. `content_9ef3d7516483e665` (client_e547b89c05043229) 89K impr, 92 clicks, pos 2.48 (top 3), CTR 0.10% vs 0.36%.
    **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: HIGH** (solid clicks, top-3 position, clean gap)
    **Would be wrong if:** branded/navigational query, same caveat as #2 and #13 — third page from this client.

16. `content_425715547c6a3ea8` (client_73cda7b4e4f265ea) 72K impr, **3 clicks**, pos 6.4, CTR 0.004% vs 0.32%.
    **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: LOW** (3 clicks — very fragile, fourth page from this client)
    **Would be wrong if:** same technical root cause as #3/#5 from this same client, not 4 independent issues.

17. `content_36fc1ee501ec072d` (client_62f4a7e64f5e0096) 73K impr, 16 clicks, pos 6.46, CTR 0.02% vs 0.32%.
    **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: MEDIUM** (16 clicks, thin sixth page from this client)
    **Would be wrong if:** client-wide template issue rather than page-specific problems.

18. `content_e578ac84778da489` (client_73cda7b4e4f265ea) 118K impr, 163 clicks, pos 4.12, CTR 0.14% vs 0.32%.
    **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: HIGH** (good click count, fifth page from this client)
    **Would be wrong if:** systemic client-side issue, same caveat repeated across this client's pages.

19. `content_c46df0fa61530d86` (client_e547b89c05043229) 70K impr, 42 clicks, pos 1.56 (top 3), CTR 0.06% vs 0.36%.
    **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: HIGH** (top-3 position, low CTR, fourth page from this client)
    **Would be wrong if:** branded query pattern repeats across this client worth checking as a group, not 4 separate fixes.

20. `content_4977e90c4d93cf9f` (client_e5c2aa26a8598242) 74K impr, 28 clicks, pos 7.4, CTR 0.04% vs 0.32%.
    **Action:** review_title_and_meta | **Reason:** ctr_review_candidate | **Confidence: MEDIUM** (28 clicks, moderate stability, different client confirms pattern isn't limited to the 3 dominant clients above)
    **Would be wrong if:** small-sample noise at this click count.

**Note on client concentration (top 20):** `client_62f4a7e64f5e0096` appears 6 times,
`client_73cda7b4e4f265ea` appears 5 times, and `client_e547b89c05043229` appears 4 times together these 3 clients account for **15 of the top 20** flagged pages. This strongly
suggests either (a) a genuine systemic issue for these specific clients worth flagging as
a pattern, or (b) these clients simply generate far more traffic volume than others, so
they mechanically dominate an impressions-weighted score. This should be verified by
comparing each client's *average* CTR gap (not raw score) against the overall average
before treating it as a client-specific finding.

**Note on confidence:** 5 of the 20 entries (#3, #5, #11, #16, and to a lesser extent #9)
rest on very low click counts (1–6 clicks). These are marked LOW confidence the score
formula is mathematically valid, but a single extra or missing click would meaningfully
change their CTR, so they should get a manual look rather than automatic action.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
q_cols = f"""
DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(q_cols).df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


**Weak picks**

- `content_acbcc847f8996314` (rank #8, client_62f4a7e64f5e0096) has the smallest CTR gap
  of the top 10 (0.15% vs 0.32% expected) it's flagged mainly because of high
  impressions (171K), not because its underperformance is severe. It's the weakest
  case for immediate action; worth a second look before prioritizing over others.

- `content_8e1334d6356668e3` and `content_fec55986a1868d62` (both client_73cda7b4e4f265ea,
  ranks #3 and #5) each had only 1 click across the whole month. With so few clicks,
  a single accidental or bot click either way would swing the CTR substantially the
  score is technically valid but statistically fragile at this click count, and it's
  worth flagging that near-zero-click pages deserve a manual look rather than blind
  trust in the score.

- Five of the top 10 belong to `client_62f4a7e64f5e0096`. As noted above, this may
  reflect a real client-level issue or may simply be a volume effect treating each of
  these 5 as independently ranked findings could overstate how many distinct problems
  actually exist.

**Leakage check**

Checked all 31 columns in `fact_content_daily_performance` (via `DESCRIBE`) this table
contains only raw traffic/engagement metrics (GSC clicks/impressions/position, GA4
sessions/engagement, channel breakdowns, AI referral counts). No flag, label, or
FlyRank-decision columns are present in this table, so none could have leaked into the
score.

The only other table used was `dim_content`, and only for `content_updated_date` (staleness
check, Signal 1) `optimization_eligible_date` and `last_optimized_date`, which do reflect
FlyRank's own product decisions, were seen in the schema but deliberately **not** used
anywhere in the Section 2 scoring query. Confirmed: no product flags, no future-dated
outcome columns, and no post-decision fields were used to build the rule or the score.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.